# 05 — Stance Detection Baseline (FNC-1 Dataset)

**Goal:** Build a lightweight TF-IDF + Logistic Regression baseline for  
headline–body **stance detection** using the [Fake News Challenge (FNC-1)](http://www.fakenewschallenge.org/) dataset.

### What is Stance Detection?

Given a **headline** and an **article body**, classify the relationship between them:

| Label | Meaning |
|-------|---------|
| **Support** | The body agrees with / supports the headline |
| **Against** | The body disagrees with / refutes the headline |
| **Neutral** | The body discusses the topic but takes no clear side, or is unrelated |

### How does this differ from Fake News Detection?

| | Fake News Detection | Stance Detection |
|---|---|---|
| Input | A single article | A headline + body pair |
| Output | Fake or Real | Support / Against / Neutral |
| Task | Classify credibility | Classify relationship |
| Use case | End-user tool | Fact-checking pipeline step |

Stance detection is a **building block** for fake news detection —  
if many credible sources *disagree* with a claim, that claim is more likely false.

> **Note:** This is a supplementary experiment only. The main project uses DistilBERT for fake news classification.

## 0. Imports

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

pd.set_option("display.max_colwidth", 120)

---
## 1. Load & Merge Data

In [ ]:
stances = pd.read_csv("train_stances.csv")
bodies  = pd.read_csv("train_bodies.csv")

print(f"Stances: {stances.shape}  |  Columns: {stances.columns.tolist()}")
print(f"Bodies:  {bodies.shape}   |  Columns: {bodies.columns.tolist()}")

In [ ]:
# Merge on Body ID to get headline + body + stance in one row
df = stances.merge(bodies, on="Body ID", how="left")

print(f"Merged shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nMissing values:\n{df.isnull().sum()}")
df.head(3)

In [ ]:
print("Original stance distribution:")
print(df["Stance"].value_counts())

fig, ax = plt.subplots(figsize=(6, 3.5))
df["Stance"].value_counts().plot(
    kind="bar", ax=ax, rot=0,
    color=["#3498db", "#2ecc71", "#e74c3c", "#f39c12"],
)
ax.set_title("Original FNC-1 Stance Distribution")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

print("\nNote: The dataset is heavily imbalanced — 'unrelated' dominates.")

---
## 2. Label Simplification

The original FNC-1 has 4 classes. We simplify to 3 for a clearer analysis:

| Original | Simplified | Reason |
|----------|------------|--------|
| agree | **Support** | Body supports the headline |
| disagree | **Against** | Body contradicts the headline |
| discuss | **Neutral** | Related but no clear stance |
| unrelated | **Neutral** | Not related — also no stance |

In [ ]:
LABEL_MAP = {
    "agree":     "Support",
    "disagree":  "Against",
    "discuss":   "Neutral",
    "unrelated": "Neutral",
}

df["label"] = df["Stance"].map(LABEL_MAP)

print("Simplified label distribution:")
print(df["label"].value_counts())

fig, ax = plt.subplots(figsize=(5, 3.5))
df["label"].value_counts().plot(
    kind="bar", ax=ax, rot=0,
    color=["#95a5a6", "#2ecc71", "#e74c3c"],
)
ax.set_title("Simplified Stance Distribution (3 classes)")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

---
## 3. Preprocessing

Combine headline and body into a single text field with a `[SEP]` separator,  
similar to how BERT-style models handle sentence pairs.

In [ ]:
def clean_text(text):
    """Basic text cleaning."""
    text = str(text).lower().strip()
    text = re.sub(r"https?://\S+", "", text)       # remove URLs
    text = re.sub(r"\s+", " ", text)                # collapse whitespace
    return text


df["headline_clean"] = df["Headline"].apply(clean_text)
df["body_clean"]     = df["articleBody"].apply(clean_text)

# Combine with [SEP] separator
df["content"] = df["headline_clean"] + " [SEP] " + df["body_clean"]

print(f"Sample combined text (first 200 chars):")
print(df["content"].iloc[0][:200])
print(f"\nContent length stats:")
print(df["content"].str.len().describe().to_string())

---
## 4. Train / Test Split

In [ ]:
X = df["content"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")
print(f"\nTrain distribution:\n{y_train.value_counts().to_string()}")
print(f"\nTest distribution:\n{y_test.value_counts().to_string()}")

---
## 5. TF-IDF Feature Extraction

In [ ]:
tfidf = TfidfVectorizer(
    max_features=10_000,
    ngram_range=(1, 2),
    stop_words="english",
    min_df=3,
    max_df=0.95,
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print(f"TF-IDF matrix shape: {X_train_tfidf.shape}")

---
## 6. Logistic Regression (Multi-Class)

In [ ]:
model = LogisticRegression(
    max_iter=1000,
    C=1.0,
    multi_class="multinomial",
    solver="lbfgs",
    random_state=42,
)

model.fit(X_train_tfidf, y_train)
y_pred = model.predict(X_test_tfidf)

print("Model training complete.")

---
## 7. Evaluation

In [ ]:
labels_order = ["Support", "Against", "Neutral"]

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average="macro")
rec  = recall_score(y_test, y_pred, average="macro")
f1   = f1_score(y_test, y_pred, average="macro")

print("=" * 50)
print("   STANCE DETECTION — TEST RESULTS")
print("=" * 50)
print(f"  Accuracy       : {acc:.4f}")
print(f"  Precision (macro): {prec:.4f}")
print(f"  Recall    (macro): {rec:.4f}")
print(f"  F1 Score  (macro): {f1:.4f}")
print("=" * 50)
print(f"\n{classification_report(y_test, y_pred, labels=labels_order)}")

---
## 8. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=labels_order)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Raw counts
ConfusionMatrixDisplay(cm, display_labels=labels_order).plot(
    ax=axes[0], cmap="Blues", values_format="d"
)
axes[0].set_title("Confusion Matrix (counts)")

# Normalized (percentage per true class)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
ConfusionMatrixDisplay(cm_norm, display_labels=labels_order).plot(
    ax=axes[1], cmap="Oranges", values_format=".2%"
)
axes[1].set_title("Confusion Matrix (normalized)")

plt.suptitle("Stance Detection — TF-IDF + Logistic Regression", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 9. Per-Class Analysis

In [ ]:
# Show top TF-IDF features for each class
feature_names = np.array(tfidf.get_feature_names_out())
top_k = 10

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = {"Support": "#2ecc71", "Against": "#e74c3c", "Neutral": "#3498db"}

for i, cls in enumerate(labels_order):
    cls_idx = list(model.classes_).index(cls)
    coefs = model.coef_[cls_idx]
    top_idx = np.argsort(coefs)[-top_k:][::-1]

    axes[i].barh(range(top_k), coefs[top_idx][::-1], color=colors[cls])
    axes[i].set_yticks(range(top_k))
    axes[i].set_yticklabels(feature_names[top_idx][::-1])
    axes[i].set_xlabel("Coefficient")
    axes[i].set_title(f"Top {top_k} → {cls}")

plt.suptitle("Most Discriminative Features per Stance Class", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 10. Quick Inference Examples

In [ ]:
examples = [
    {
        "headline": "New study confirms vaccines are safe and effective",
        "body": "Researchers at Johns Hopkins University published findings that confirm the safety and efficacy of current vaccines. The study reviewed data from over 10 million patients.",
    },
    {
        "headline": "City to ban all cars from downtown area",
        "body": "The mayor's office denied rumors of a car ban. Officials stated that no such policy is under consideration and traffic will continue as normal.",
    },
    {
        "headline": "Tech company announces record profits",
        "body": "A new species of frog was discovered in the Amazon rainforest by a team of biologists from Brazil. The frog has a unique blue coloration.",
    },
]

print(f"{'Headline':<50} {'Predicted Stance':>16}")
print("-" * 70)

for ex in examples:
    combined = clean_text(ex["headline"]) + " [SEP] " + clean_text(ex["body"])
    vec  = tfidf.transform([combined])
    pred = model.predict(vec)[0]
    print(f"{ex['headline'][:48]:<50} → {pred:>10}")

---
## Summary

| Item | Detail |
|------|--------|
| Dataset | FNC-1 (`train_stances.csv` + `train_bodies.csv`) |
| Samples | 49,972 headline–body pairs |
| Labels | Support / Against / Neutral (simplified from 4 → 3) |
| Features | TF-IDF (10k features, unigrams + bigrams) |
| Model | Logistic Regression (multinomial) |
| Split | 80% train / 20% test (stratified) |

### Why TF-IDF Instead of a Transformer?

| Reason | Detail |
|--------|--------|
| **Speed** | Trains in seconds vs. minutes/hours |
| **Simplicity** | Easy to explain in a presentation |
| **Baseline** | Establishes a performance floor before trying complex models |
| **Scope** | This is a supplementary experiment, not the main project |

### Limitations

- TF-IDF ignores word order and semantic meaning
- The "Against" class has very few samples → low recall
- Merging "discuss" and "unrelated" into "Neutral" loses nuance
- A transformer model would likely improve performance, especially on minority classes